# Lab W5 (WAJIB): RNN vs LSTM vs GRU pada Sequence

## Pre-flight Checklist

> [!IMPORTANT]
> Lab ini **wajib** untuk W5. Memenuhi Breadth Check keluarga RNN/LSTM (lihat Kontrak Belajar §6 di Pendahuluan). Konsep yang ditandai (§) merujuk ke `05_W5_Sequences_RNN_LSTM.md`.

**Yang Anda butuhkan sebelum mulai:**
- Bab W5 sudah dibaca, terutama §1.5 (BPTT primer + tabel vanishing numerik), §2.2 (RNN vanilla annotated), §2.3 (LSTM gates + Hadamard primer + cara kerja forget gate + hidden vs cell state).
- Familiar dengan tuple shape `(B, T, F)` untuk batch sequence (lihat §0.5.1 Pendahuluan).
- Familiar dengan gradient clipping (`torch.nn.utils.clip_grad_norm_`).

**Yang akan Anda hasilkan di akhir lab:**
- Smoke test untuk RNN vanilla, LSTM, dan GRU (overfit satu batch, loss < 0.01).
- Plot gradient norm log-scale: vanilla RNN turun eksponensial, LSTM relatif flat (bukti vanishing gradient §1.5.2).
- Training 3 arsitektur (RNN, LSTM, GRU) pada seq_len=50 dan seq_len=200.
- Tabel perbandingan MAE: RNN vs LSTM vs GRU pada kedua sequence length.
- Visualisasi prediksi vs ground truth pada 6 sampel validasi per arsitektur.
- Justifikasi arsitektur berdasarkan hasil eksperimen.
- Refleksi yang menghubungkan eksperimen ke teori vanishing/exploding gradient.

**Resource:**
- **Hardware:** CPU cukup (sequence pendek, model kecil). GPU mempercepat tetapi tidak wajib.
- **Estimasi waktu kerja:** 4-6 jam termasuk membaca, eksekusi, dan refleksi.

**Pendamping:** Bab W5 di `05_W5_Sequences_RNN_LSTM.md`.

**Tujuan pedagogis:** mendemonstrasikan secara visual perbedaan gradient flow antara vanilla RNN, LSTM, dan GRU; melatih ketiganya untuk one-step-ahead prediction pada sinyal sine + noise sintetis di dua sequence length; menulis justifikasi arsitektur berdasarkan bukti eksperimental.

## 1. Setup

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
# Di Google Colab: clone repo terlebih dahulu, lalu cd ke template_repo/
#   git clone <url-repo> && cd template_repo
# Di environment lokal: jalankan dari folder notebooks/
import sys, os
_root = os.path.abspath('..')
if _root not in sys.path:
    sys.path.insert(0, _root)
print("Root:", _root)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

from src.models import SimpleLSTM

torch.manual_seed(42)
np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 2. Dataset sintetis: sine + noise

Dataset forecasting sederhana — diberi N langkah sebelumnya, prediksi langkah ke-N+1. Kita siapkan dua versi: seq_len=50 dan seq_len=200 untuk menguji efek panjang sequence pada vanishing gradient.

In [ ]:
def make_sine_dataset(n_samples=4000, seq_len=50, noise=0.05, seed=42):
    rng = np.random.default_rng(seed)
    starts = rng.uniform(0, 2 * np.pi, size=n_samples)
    freqs = rng.uniform(0.5, 2.0, size=n_samples)
    t = np.linspace(0, 4 * np.pi, seq_len + 1)
    data = np.sin(starts[:, None] + freqs[:, None] * t)  # (n, seq_len+1)
    data += rng.normal(0, noise, data.shape)
    X = data[:, :-1][:, :, None].astype(np.float32)      # (n, seq_len, 1)
    y = data[:, -1:].astype(np.float32)                  # (n, 1)
    return X, y

# Dataset seq_len=50
X_tr, y_tr = make_sine_dataset(4000, 50, noise=0.05, seed=42)
X_va, y_va = make_sine_dataset(500, 50, noise=0.05, seed=1337)
print('seq_len=50 | X train:', X_tr.shape, 'y train:', y_tr.shape)

plt.plot(X_tr[0, :, 0]); plt.scatter([50], y_tr[0], color='red', label='target'); plt.legend(); plt.title('Satu sampel seq_len=50')
plt.show()

In [ ]:
# Dataset seq_len=200 — untuk menguji efek panjang sequence yang lebih ekstrem
X_tr_200, y_tr_200 = make_sine_dataset(4000, 200, noise=0.05, seed=42)
X_va_200, y_va_200 = make_sine_dataset(500, 200, noise=0.05, seed=1337)
print('seq_len=200 | X train:', X_tr_200.shape, 'y train:', y_tr_200.shape)

plt.plot(X_tr_200[0, :, 0]); plt.scatter([200], y_tr_200[0], color='red', label='target'); plt.legend(); plt.title('Satu sampel seq_len=200')
plt.show()

## 3. Smoke Test: Overfit Satu Batch

Sebelum training penuh, buktikan bahwa gradient mengalir untuk ketiga arsitektur. Strategi: 8 sampel seq_len=50, 200 iterasi, loss target < 0.01. Kalau gagal di sini, semua training berikutnya sia-sia.

In [ ]:
class VanillaRNN(nn.Module):
    """Vanilla RNN wrapper (tanpa LSTM gates) — untuk perbandingan eksplisit."""
    def __init__(self, input_size=1, hidden_size=64, num_layers=1, num_classes=1):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.head = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out[:, -1, :])

class LSTMv2(nn.Module):
    """LSTM wrapper untuk perbandingan setara."""
    def __init__(self, input_size=1, hidden_size=64, num_layers=1, num_classes=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.head = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])

class GRUModel(nn.Module):
    """GRU wrapper untuk perbandingan setara."""
    def __init__(self, input_size=1, hidden_size=64, num_layers=1, num_classes=1):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.head = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        out, _ = self.gru(x)
        return self.head(out[:, -1, :])

def overfit_one_batch(model_cls, name, n_iter=200, target_loss=0.01):
    """Smoke test: overfit 8 sampel, pastikan loss < target_loss."""
    torch.manual_seed(0)
    model = model_cls().to(device)
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    crit = nn.MSELoss()
    x = torch.randn(8, 50, 1).to(device)
    y = torch.randn(8, 1).to(device)
    losses = []
    for i in range(n_iter):
        opt.zero_grad()
        loss = crit(model(x), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        losses.append(loss.item())
    final = losses[-1]
    ok = final < target_loss
    status = '✓' if ok else '✗'
    print(f'{status} {name:15s}: loss {losses[0]:.4f} → {final:.4f} (target < {target_loss})')
    return final

print('Smoke test — overfit 8 sampel random, 200 iterasi…\n')
rnn_smoke = overfit_one_batch(VanillaRNN, 'Vanilla RNN')
lstm_smoke = overfit_one_batch(LSTMv2, 'LSTM')
gru_smoke = overfit_one_batch(GRUModel, 'GRU')
print('\nSemua arsitektur lolos smoke test — gradient mengalir di semua model.')

## 4. Vanilla RNN vs LSTM: Gradient Flow

Sebelum training, kita ilustrasikan *vanishing gradient* dengan backpropagate gradient dari loss di timestep terakhir ke setiap timestep input, lalu plot norm gradient per-timestep. Untuk RNN, gradient di timestep awal biasanya nyaris nol — inilah kenapa LSTM didesain.

In [ ]:
def gradient_norm_per_timestep(cell_type='RNN', seq_len=50, hidden=32):
    torch.manual_seed(0)
    if cell_type == 'RNN':
        cell = nn.RNN(1, hidden, batch_first=True)
    elif cell_type == 'GRU':
        cell = nn.GRU(1, hidden, batch_first=True)
    else:
        cell = nn.LSTM(1, hidden, batch_first=True)
    x = torch.randn(1, seq_len, 1, requires_grad=True)
    out, _ = cell(x)
    loss = out[:, -1, :].sum()
    loss.backward()
    g = x.grad[0, :, 0].abs().numpy()
    return g

# Gradient flow — seq_len=50
g_rnn  = gradient_norm_per_timestep('RNN', 50)
g_lstm = gradient_norm_per_timestep('LSTM', 50)
g_gru  = gradient_norm_per_timestep('GRU', 50)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(g_rnn,  label='vanilla RNN', marker='.')
axes[0].plot(g_lstm, label='LSTM',        marker='.')
axes[0].plot(g_gru,  label='GRU',         marker='.')
axes[0].set_yscale('log')
axes[0].set_xlabel('timestep (0 paling lama, 49 terbaru)')
axes[0].set_ylabel('|gradient| di input')
axes[0].set_title('Gradient flow — seq_len=50')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Gradient flow — seq_len=200
g_rnn_200  = gradient_norm_per_timestep('RNN', 200)
g_lstm_200 = gradient_norm_per_timestep('LSTM', 200)
g_gru_200  = gradient_norm_per_timestep('GRU', 200)

axes[1].plot(g_rnn_200,  label='vanilla RNN', marker='.')
axes[1].plot(g_lstm_200, label='LSTM',        marker='.')
axes[1].plot(g_gru_200,  label='GRU',         marker='.')
axes[1].set_yscale('log')
axes[1].set_xlabel('timestep (0 paling lama, 199 terbaru)')
axes[1].set_ylabel('|gradient| di input')
axes[1].set_title('Gradient flow — seq_len=200')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Statistik numerik
print('=== seq_len=50 ===')
print(f'RNN  | awal: {g_rnn[0]:.2e}  akhir: {g_rnn[-1]:.2e}  rasio awal/akhir: {g_rnn[0]/g_rnn[-1]:.2e}')
print(f'LSTM | awal: {g_lstm[0]:.2e}  akhir: {g_lstm[-1]:.2e}  rasio awal/akhir: {g_lstm[0]/g_lstm[-1]:.2e}')
print(f'GRU  | awal: {g_gru[0]:.2e}  akhir: {g_gru[-1]:.2e}  rasio awal/akhir: {g_gru[0]/g_gru[-1]:.2e}')
print()
print('=== seq_len=200 ===')
print(f'RNN  | awal: {g_rnn_200[0]:.2e}  akhir: {g_rnn_200[-1]:.2e}  rasio awal/akhir: {g_rnn_200[0]/g_rnn_200[-1]:.2e}')
print(f'LSTM | awal: {g_lstm_200[0]:.2e}  akhir: {g_lstm_200[-1]:.2e}  rasio awal/akhir: {g_lstm_200[0]/g_lstm_200[-1]:.2e}')
print(f'GRU  | awal: {g_gru_200[0]:.2e}  akhir: {g_gru_200[-1]:.2e}  rasio awal/akhir: {g_gru_200[0]/g_gru_200[-1]:.2e}')

## 5. Training Loop: RNN vs LSTM vs GRU

Fungsi training generik — melatih model apa pun pada dataset, mengembalikan loss history. Gradient clipping 1.0 diterapkan ke **semua** model sequence.

In [ ]:
def train_sequence_model(model, X_tr, y_tr, X_va, y_va, epochs=20, batch_size=64, lr=1e-3):
    """Training loop generik untuk model sequence.
    Returns: (hist_tr, hist_va, model_trained)
    """
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    crit = nn.MSELoss()
    ds_tr = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr))
    dl_tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=True)
    X_va_t = torch.tensor(X_va).to(device)
    y_va_t = torch.tensor(y_va).to(device)
    hist_tr, hist_va = [], []
    for epoch in range(epochs):
        model.train()
        running = 0.0
        for Xb, yb in dl_tr:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(Xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            running += loss.item() * len(Xb)
        model.eval()
        with torch.no_grad():
            va_loss = crit(model(X_va_t), y_va_t).item()
        hist_tr.append(running / len(ds_tr))
        hist_va.append(va_loss)
        print(f'{type(model).__name__:>12s} epoch {epoch+1:2d}  tr={hist_tr[-1]:.4f}  va={va_loss:.4f}')
    return hist_tr, hist_va, model

In [ ]:
print('=== Training seq_len=50 (20 epoch) ===\n')

rnn_tr_50, rnn_va_50, rnn_model = train_sequence_model(
    VanillaRNN(), X_tr, y_tr, X_va, y_va, epochs=20)

lstm_tr_50, lstm_va_50, lstm_model = train_sequence_model(
    LSTMv2(), X_tr, y_tr, X_va, y_va, epochs=20)

gru_tr_50, gru_va_50, gru_model = train_sequence_model(
    GRUModel(), X_tr, y_tr, X_va, y_va, epochs=20)

In [ ]:
# Plot perbandingan loss — seq_len=50
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (tr, va, name) in zip(axes, [
    (rnn_tr_50, rnn_va_50, 'RNN'),
    (lstm_tr_50, lstm_va_50, 'LSTM'),
    (gru_tr_50, gru_va_50, 'GRU'),
]):
    ax.plot(tr, label='train')
    ax.plot(va, label='val')
    ax.set_xlabel('epoch')
    ax.set_ylabel('MSE')
    ax.set_title(f'{name} — seq_len=50')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Tabel ringkasan
print(f'{"Model":>12s}  {"Train MSE":>10s}  {"Val MSE":>10s}  {"Val MAE":>10s}')
print('-' * 48)
for name, tr_hist, va_hist, model in [
    ('RNN', rnn_tr_50, rnn_va_50, rnn_model),
    ('LSTM', lstm_tr_50, lstm_va_50, lstm_model),
    ('GRU', gru_tr_50, gru_va_50, gru_model),
]:
    val_mse = va_hist[-1]
    # Hitung MAE pada validation set
    model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(X_va).to(device)).cpu().numpy()
        val_mae = np.abs(pred - y_va).mean()
    print(f'{name:>12s}  {tr_hist[-1]:10.4f}  {val_mse:10.4f}  {val_mae:10.4f}')

## 6. Training pada seq_len=200

Ulangi perbandingan pada sequence yang lebih panjang. Ekspektasi: RNN vanilla akan kesulitan (vanishing gradient makin parah), LSTM dan GRU tetap stabil.

In [ ]:
print('=== Training seq_len=200 (30 epoch) ===\n')

rnn_tr_200, rnn_va_200, rnn_model_200 = train_sequence_model(
    VanillaRNN(hidden_size=64), X_tr_200, y_tr_200, X_va_200, y_va_200, epochs=30)

lstm_tr_200, lstm_va_200, lstm_model_200 = train_sequence_model(
    LSTMv2(hidden_size=64), X_tr_200, y_tr_200, X_va_200, y_va_200, epochs=30)

gru_tr_200, gru_va_200, gru_model_200 = train_sequence_model(
    GRUModel(hidden_size=64), X_tr_200, y_tr_200, X_va_200, y_va_200, epochs=30)

In [ ]:
# Plot perbandingan — seq_len=200
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (tr, va, name) in zip(axes, [
    (rnn_tr_200, rnn_va_200, 'RNN'),
    (lstm_tr_200, lstm_va_200, 'LSTM'),
    (gru_tr_200, gru_va_200, 'GRU'),
]):
    ax.plot(tr, label='train')
    ax.plot(va, label='val')
    ax.set_xlabel('epoch')
    ax.set_ylabel('MSE')
    ax.set_title(f'{name} — seq_len=200')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'{"Model":>12s}  {"Train MSE":>10s}  {"Val MSE":>10s}  {"Val MAE":>10s}')
print('-' * 48)
for name, tr_hist, va_hist, model in [
    ('RNN', rnn_tr_200, rnn_va_200, rnn_model_200),
    ('LSTM', lstm_tr_200, lstm_va_200, lstm_model_200),
    ('GRU', gru_tr_200, gru_va_200, gru_model_200),
]:
    model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(X_va_200).to(device)).cpu().numpy()
        val_mae = np.abs(pred - y_va_200).mean()
    print(f'{name:>12s}  {tr_hist[-1]:10.4f}  {va_hist[-1]:10.4f}  {val_mae:10.4f}')

## 7. Tabel Perbandingan Lengkap

Ringkasan MAE untuk ketiga arsitektur pada kedua sequence length.

In [ ]:
# Hitung MAE untuk semua kombinasi
results = {}
for name, model, X, y in [
    ('RNN', rnn_model, X_va, y_va),
    ('LSTM', lstm_model, X_va, y_va),
    ('GRU', gru_model, X_va, y_va),
    ('RNN', rnn_model_200, X_va_200, y_va_200),
    ('LSTM', lstm_model_200, X_va_200, y_va_200),
    ('GRU', gru_model_200, X_va_200, y_va_200),
]:
    sl = 50 if X.shape[1] == 50 else 200
    model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(X).to(device)).cpu().numpy()
        mae = np.abs(pred - y).mean()
    results[(name, sl)] = mae

# Tabel
print(f'{"Arsitektur":>12s}  {"MAE (seq_len=50)":>16s}  {"MAE (seq_len=200)":>16s}  {"Δ MAE":>10s}')
print('-' * 60)
for name in ['RNN', 'LSTM', 'GRU']:
    mae50 = results[(name, 50)]
    mae200 = results[(name, 200)]
    delta = mae200 - mae50
    print(f'{name:>12s}  {mae50:16.4f}  {mae200:16.4f}  {delta:+10.4f}')

### Analisis Tabel Perbandingan

Ekspektasi berdasarkan teori (W5 §1.5 + §2.3):

- **RNN vanilla:** MAE di seq_len=50 mungkin setara dengan LSTM/GRU (sequence-50 masih manageable untuk RNN). Namun di seq_len=200, MAE akan memburuk secara signifikan — vanishing gradient mencegah RNN memanfaatkan informasi dari early timestep.
- **LSTM:** MAE di kedua seq_len harus rendah dan stabil. Cell state highway memutus rantai vanishing gradient, sehingga LSTM bisa memanfaatkan long-range dependency.
- **GRU:** Performa mirip LSTM (gate reset-update juga memutus vanishing chain) dengan parameter lebih sedikit. Pada dataset kecil seperti ini, GRU kadang lebih efisien.
- **Δ MAE (200→50):** RNN akan punya selisih positif terbesar (makin panjang = makin buruk). LSTM dan GRU akan punya selisih minimal (stabil di kedua panjang).

## 8. Visualisasi Prediksi

6 sampel validasi per arsitektur — prediksi vs ground truth.

In [ ]:
def plot_predictions(models_dict, X_va, y_va, n_samples=6):
    fig, axes = plt.subplots(len(models_dict), n_samples, figsize=(n_samples*2.5, len(models_dict)*2.5))
    if len(models_dict) == 1:
        axes = axes[np.newaxis, :]
    for row, (name, model) in enumerate(models_dict.items()):
        model.eval()
        with torch.no_grad():
            preds = model(torch.tensor(X_va[:n_samples]).to(device)).cpu().numpy()
        for col in range(n_samples):
            ax = axes[row, col]
            ax.plot(X_va[col, :, 0], color='steelblue', alpha=0.7, linewidth=1)
            ax.axvline(X_va.shape[1], color='gray', linestyle='--', alpha=0.5)
            ax.scatter([X_va.shape[1]], preds[col], color='orange', s=60, marker='D', label='pred', zorder=5)
            ax.scatter([X_va.shape[1]], y_va[col], color='red', s=60, marker='o', label='true', zorder=5)
            if row == 0:
                ax.set_title(f'Sample {col+1}')
            if col == 0:
                ax.set_ylabel(name)
            if row == 0 and col == 1:
                ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()

models_50 = {'RNN': rnn_model, 'LSTM': lstm_model, 'GRU': gru_model}
print('Prediksi — seq_len=50')
plot_predictions(models_50, X_va, y_va, n_samples=6)

models_200 = {'RNN': rnn_model_200, 'LSTM': lstm_model_200, 'GRU': gru_model_200}
print('\nPrediksi — seq_len=200')
plot_predictions(models_200, X_va_200, y_va_200, n_samples=6)

## 9. Justifikasi Arsitektur

Berdasarkan hasil eksperimen, tulis justifikasi arsitektur mengikuti template dari `05_W5_Sequences_RNN_LSTM.md` §2.6. Jawab pertanyaan-pertanyaan berikut:

1. **Apakah hasil konsisten dengan teori vanishing gradient?** Bandingkan rasio gradient di §4 dengan performa MAE di §5-7. Apakah arsitektur dengan gradient flow yang lebih sehat (LSTM/GRU) juga punya MAE lebih baik?

2. **Kapan selisih RNN vs LSTM paling terlihat?** Bandingkan Δ MAE di seq_len=50 vs seq_len=200. Pada panjang mana RNN mulai tertinggal secara signifikan?

3. **LSTM vs GRU:** Mana yang lebih baik di dataset ini? Faktor apa (parameter count, training speed, MAE) yang menentukan pilihan?

4. **Jika Anda harus merekomendasikan satu arsitektur** untuk forecasting time-series dengan panjang sequence ≤50: mana yang Anda pilih dan mengapa?

Tulis justifikasi Anda di sel di bawah:

### Jawaban Justifikasi Arsitektur

**1. Konsistensi dengan teori vanishing gradient:**

> *[tulis di sini]*

**2. Kapan selisih RNN vs LSTM paling terlihat:**

> *[tulis di sini]*

**3. LSTM vs GRU:**

> *[tulis di sini]*

**4. Rekomendasi arsitektur untuk sequence ≤50:**

> *[tulis di sini]*

## 10. Refleksi

### 10.1 Pertanyaan Tertarget

1. **Vanishing gradient quantified.** Pada cell §4, hitung rasio `g_rnn[0] / g_rnn[-1]` dan `g_lstm[0] / g_lstm[-1]` untuk seq_len=50 dan seq_len=200. Apakah hasil Anda konsisten dengan tabel `w_h^T` di §1.5.2 W5? Pilih jawaban paling akurat:
   - (a) Rasio RNN < 1e-3, rasio LSTM ~ orde-1. Konsisten dengan vanishing gradient.
   - (b) Rasio RNN dan LSTM keduanya ~ orde-1. Vanishing gradient tidak terlihat (perlu seq_len lebih panjang).
   - (c) Rasio RNN ~ orde-1, rasio LSTM < 1e-3. Hasil tidak sesuai prediksi — ada bug di setup.
   - (d) Hasil bervariasi antar run.

2. **Gradient clipping eksperimen.** Komen-out `torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)` di fungsi `train_sequence_model`, latih ulang salah satu arsitektur. Apakah loss tetap stabil, atau muncul lonjakan/NaN? Pada titik mana (epoch berapa) gejala muncul? Hubungkan dengan §1.5.2 W5 (rezim `|w_h| > 1` exploding).

3. **MLP baseline.** Ganti model sequence dengan `SimpleMLP(input_dim=seq_len, hidden_sizes=(64,), num_classes=1)` dan flatten input `Xb.flatten(start_dim=1)`. Latih 20 epoch pada seq_len=50. Apakah val MAE sebanding dengan RNN/LSTM/GRU? Hubungkan dengan §2.2 W5 (RNN family vs alternatives) dan strategi representasi W3 §2.4.

### 10.2 Self-Check Quick

- [ ] Smoke test 3 arsitektur lolos (loss < 0.01).
- [ ] Plot gradient flow menunjukkan vanishing pada RNN, flat pada LSTM/GRU.
- [ ] Training semua arsitektur selesai di seq_len=50 dan seq_len=200.
- [ ] Tabel perbandingan MAE lengkap.
- [ ] Justifikasi arsitektur ditulis berdasarkan data (bukan opini).
- [ ] Visualisasi prediksi 6 sampel menunjukkan perbedaan kualitas antar arsitektur.
- [ ] Saya bisa menjelaskan kenapa cell state di LSTM memutus rantai vanishing gradient.
- [ ] Gradient clipping aktif di semua training loop.